In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from pathlib import Path
from functools import partial
import json

import time
import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10.95})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
use_f64 = False  # False

import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", use_f64)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result,
    evaluate_model_prediction, evaluate_model_rollout, evaluate_persistence_prediction
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts
from admpe.sets import SafelyExcitableSet
from admpe.evaluation.modeling_evaluation import resolve_dataset_path


---

In [ ]:
from admpe.data_management import QUANTITATIVE_RESULTS_ROOT

In [ ]:
EXPERIMENTS = [
    #"submission_nov_1step",
    "submission_nov_curv64_1step", #"submission_nov_curv64", # 
]

MODEL_TYPES = [
    "neural_euler_large_1", #"gp_transition", # 
]

# CONFIG = {
#     "version": 1,
#     "points_per_dim": {
#         "fluid_tank": 50,  # Existing density support: 50.
#         "pendulum": 21,     # Default density support: 21.
#         "cart_pole": 10,    # Existing density support: 10.
#         "pmsm_nominal": 21,
#     },
#     "bounds": [-1.0, 1.0],
#     "evaluation_dtype": "float64" if use_f64 else "float32",
# }

# CONFIG = {
#     "version": 2,
#     "points_per_dim": {
#         "fluid_tank": 100,  # Existing density support: 50.
#         "pendulum": 42,     # Default density support: 21.
#         "cart_pole": 20,    # Existing density support: 10.
#         "pmsm_nominal": 42,
#     },
#     "bounds": [-1.0, 1.0],
#     "evaluation_dtype": "float64" if use_f64 else "float32",
# }

CONFIG = {
    "version": 3,
    "points_per_dim": {
        "fluid_tank": 200,
        "pendulum": 84,
        "cart_pole": 28,
        "pmsm_nominal": 50,
    },
    "bounds": [-1.0, 1.0],
    "evaluation_dtype": "float32",
}

# CONFIG = {
#     "version": 4,
#     "points_per_dim": {
#         "fluid_tank": 200,
#         "pendulum": 84,
#         "cart_pole": 28,
#         "pmsm_nominal": 50,
#     },
#     "bounds": [-1.0, 1.0],
#     "evaluation_dtype": "float64",
# }

filter_testing_points = True
tag = "safely_excitable" if filter_testing_points else "full_box"
show_baseline = False

In [ ]:
runs = [
    (experiment, entry)
    for experiment in EXPERIMENTS
    for model_type in MODEL_TYPES
    for entry in list_modeling_results(
        exp_name=experiment,
        model_type=model_type,
        status="completed",
    )
]

name = f"one-step-modeling-results-{'-'.join(EXPERIMENTS)}-{tag}-v{CONFIG["version"]}"

output_path = (
    QUANTITATIVE_RESULTS_ROOT
    / f"{name}.parquet"
)

print(jax.devices())
print(f"{len(runs)} completed models")

In [ ]:
def grid_blocks(dim, points_per_dim, block_size):
    if points_per_dim < 2:
        raise ValueError("Need at least two grid points per dimension.")

    shape = (points_per_dim,) * dim
    total = points_per_dim**dim

    for start in range(0, total, block_size):
        flat = np.arange(start, min(start + block_size, total))
        indices = np.stack(np.unravel_index(flat, shape), axis=-1)

        yield -1.0 + indices * (2.0 / (points_per_dim - 1))

In [ ]:
@eqx.filter_jit
def prediction_block(model, featurize, points, obs_dim, tau, batch_size):
    def predict(point):
        obs, action = point[:obs_dim], point[obs_dim:]
        return featurize(model.step(obs, action, tau))

    return jax.lax.map(predict, points, batch_size=batch_size)


@eqx.filter_jit
def reference_block(env, featurize, points, batch_size):
    def reference(point):
        obs, action = point[:env.obs_dim], point[env.obs_dim:]
        state = env.generate_state_from_observation(obs)
        next_obs, _ = env.step(state, action)

        # Ground truth and persistence prediction in the same feature space.
        return featurize(next_obs), featurize(obs)

    return jax.lax.map(reference, points, batch_size=batch_size)


class MetricSums:
    def __init__(self):
        self.n = self.bad = 0
        self.sse = self.sae = self.wce = 0.0

    def update(self, prediction, reference):
        prediction, reference = np.asarray(prediction), np.asarray(reference)

        if prediction.shape != reference.shape or prediction.ndim != 2:
            raise ValueError("Expected matching (n_points, n_features) arrays.")
        if not np.isfinite(reference).all():
            raise FloatingPointError("Nonfinite reference transition.")

        error = prediction - reference
        absolute_error = np.abs(error)

        self.n += len(error)
        self.bad += np.count_nonzero(
            ~np.isfinite(prediction).all(axis=-1)
        )
        self.sse = self.sse + np.sum(
            error**2, axis=0, dtype=jnp.float32 if not use_f64 else jnp.float64
        )
        self.sae = self.sae + np.sum(
            absolute_error, axis=0, dtype=jnp.float32 if not use_f64 else jnp.float64
        )
        self.wce = np.maximum(
            self.wce, np.max(absolute_error, axis=0)
        )

    def finish(self):
        if self.n == 0:
            raise ValueError("No evaluation points were accepted.")

        return {
            "rmse": float(np.sqrt(np.mean(self.sse / self.n))),
            "mae": float(np.mean(self.sae / self.n)),
            "wce": float(np.max(self.wce)),
            "rmse_per_feature": np.sqrt(self.sse / self.n).tolist(),
            "mae_per_feature": (self.sae / self.n).tolist(),
            "wce_per_feature": self.wce.tolist(),
            "nonfinite_prediction_fraction": float(self.bad / self.n),
        }

In [ ]:
def evaluate_dense_one_step(
    wrapped_model,
    wrapped_env,
    points_per_dim,
    *,
    model_tau=1.0,
    safe_set=None,
    block_size=4096,
    batch_size=64,
):
    model_stats = MetricSums()
    baseline_stats = MetricSums()
    env = wrapped_env.model

    for points in grid_blocks(
        env.obs_dim + env.action_dim,
        points_per_dim,
        block_size,
    ):
        if safe_set is not None:
            keep = np.asarray(
                safe_set.contains(jnp.asarray(points))
            )
            points = points[keep]

        n = len(points)
        if n == 0:
            continue

        # Keep prediction shapes fixed; exclude padding from every metric.
        padded = np.pad(
            points,
            ((0, block_size - n), (0, 0)),
            mode="edge",
        )
        padded = jnp.asarray(padded, dtype=jnp.float32 if not use_f64 else jnp.float64)

        reference, persistence = reference_block(
            env, wrapped_env.featurize, padded, batch_size
        )
        prediction = prediction_block(
            wrapped_model.model,
            wrapped_model.featurize,
            padded,
            env.obs_dim,
            model_tau,
            batch_size,
        )

        # Synchronizes and transfers only this block to the host.
        prediction, reference, persistence = (
            np.asarray(array)[:n]
            for array in (prediction, reference, persistence)
        )

        model_stats.update(prediction, reference)
        baseline_stats.update(persistence, reference)

    return {
        "metrics": model_stats.finish(),
        "baseline_metrics": baseline_stats.finish(),
        "n_points": model_stats.n,
    }

In [ ]:
# saved = pd.read_parquet(output_path)
# print(f"{len(saved)} models saved")
# display(saved.tail())

In [ ]:
output_path

In [ ]:
df = (
    pd.read_parquet(output_path)
    if output_path.exists()
    else pd.DataFrame()
)
# df = pd.DataFrame()
seen = set(df["exp_id"]) if not df.empty else set()
safe_sets = {}

for experiment, entry in tqdm.tqdm(runs):
    if entry["name"] in seen:
        continue

    result = load_modeling_result(
        entry["path"], dtype=jnp.float32 if not use_f64 else jnp.float64
    )
    metadata = result["metadata"]
    system = metadata["system_name"]


    
    dataset_path = resolve_dataset_path(metadata["dataset_path"])
    excitation_metadata = json.loads(
        (dataset_path / "experiment.json").read_text()
    )

    wrapped_model, wrapped_env = get_wrapped_model_and_env(result)

    if filter_testing_points:
        if system not in safe_sets:
            safe_sets[system] = SafelyExcitableSet.from_system_name(system)
        safe_set, safe_set_path = safe_sets[system]
    else:
        safe_set, safe_set_path = None, None

    model_tau = float(
        metadata["params"]["model_trainer_params"].get("tau", 1.0)
    )
    points_per_dim = CONFIG["points_per_dim"][system]

    evaluation = evaluate_dense_one_step(
        wrapped_model,
        wrapped_env,
        points_per_dim,
        model_tau=model_tau,
        safe_set=safe_set,
    )

    row = {
        "exp_id": metadata["exp_id"],
        "path": str(entry["path"]),
        "modeling_experiment": experiment,
        "system_name": system,
        "model_type": metadata["model_type"],
        "training_seed": metadata["seed"],
        "use_f64": metadata.get("use_f64"),
        "dataset_path": str(dataset_path),
        "excitation_seed": excitation_metadata["seed"],
        "excitation_objective": excitation_metadata["excitation_objective"],
        "evaluation_dtype": CONFIG["evaluation_dtype"],
        "filter_testing_points": filter_testing_points,
        "safe_set_path": str(safe_set_path) if safe_set_path else None,
        "grid_points_per_dim": points_per_dim,
        "n_candidate_points": points_per_dim ** (
            wrapped_env.model.obs_dim + wrapped_env.model.action_dim
        ),
        "n_prediction_points": evaluation["n_points"],
        "model_tau": model_tau,
        "environment_tau": float(wrapped_env.model.tau),
        "prediction": {"metrics": evaluation["metrics"]},
        "baseline": {"metrics": evaluation["baseline_metrics"]},
    }

    df = pd.concat(
        [df, pd.json_normalize([row], sep=".")],
        ignore_index=True,
    )
    df.to_parquet(output_path, index=False)
    seen.add(metadata["exp_id"])

    del result, wrapped_model, wrapped_env

## Visualize:

In [ ]:
from admpe.evaluation.modeling_visualization import (
    visualize_model_prediction_metric_boxplots,
)

figures = visualize_model_prediction_metric_boxplots(
    df,
    log_scale=True,
    figure_width=10,
    hide_outliers=False,
)

for system, (fig, axes) in figures.items():
    baseline = df.loc[df["system_name"].eq(system)].iloc[0]

    # for ax, metric in zip(axes, ("rmse", "mae", "wce")):
    #     ax.axhline(
    #         baseline[f"baseline.metrics.{metric}"],
    #         color="black",
    #         linestyle="--",
    #         label="Persistence",
    #     )

    # axes[0].legend()
    display(fig)